# Preparação de dados meteorológicos multivariados

Este notebook prepara os dados horários da estação INMET A707 para previsão direta multialvo de **temperatura do ar** e **umidade relativa** nas próximas 24 horas.

## Protocolo

- Histórico de entrada: 24 horas.
- Saída: `t+1` a `t+24` para temperatura e umidade.
- Treino: 2011 a 2023, excluindo 2022.
- Validação: 2024.
- Teste: 2025.
- Avaliação adicional: 2026 parcial.
- Covariáveis futuras: somente atributos temporais determinísticos.
- Sem imputação dos alvos ou das entradas meteorológicas.
- Normalizadores ajustados exclusivamente com dados de treino.

O notebook gera artefatos separados para os conjuntos **TU**, **TUO** e **TUOV**.

## 1. Importações e configuração geral

Altere apenas `CAMINHO_DADOS` e `DIRETORIO_SAIDA` se a organização local for diferente.

In [3]:
from pathlib import Path
import json
import hashlib
import platform
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

CAMINHO_DADOS = Path("dados_inmet_consolidados_2026-09-20_16-30.csv")
DIRETORIO_SAIDA = Path("dados_processados_multivariados")

TAU = 24
HORIZONTE = 24

ANOS_TREINO = [ano for ano in range(2011, 2024) if ano != 2022]
ANOS_EXCLUIDOS = [2022]
ANO_VALIDACAO = 2024
ANO_TESTE = 2025
ANO_AVALIACAO_ADICIONAL = 2026

COLUNAS_ALVO = [
    "temperatura_ar_horaria_c",
    "umidade_horaria_pct",
]

CONJUNTOS = {
    "TU": [
        "temperatura_ar_horaria_c",
        "umidade_horaria_pct",
    ],
    "TUO": [
        "temperatura_ar_horaria_c",
        "umidade_horaria_pct",
        "temperatura_ponto_orvalho_c",
    ],
    "TUOV": [
        "temperatura_ar_horaria_c",
        "umidade_horaria_pct",
        "temperatura_ponto_orvalho_c",
        "vento_velocidade_horaria_ms",
    ],
}

COLUNAS_TEMPORAIS = [
    "hora_sin",
    "hora_cos",
    "dia_ano_sin",
    "dia_ano_cos",
]

COLUNAS_FUTURAS_PERMITIDAS = COLUNAS_TEMPORAIS.copy()

CONTAGENS_ESPERADAS = {
    "TU": {"treino": 84744, "validacao": 7837, "teste": 7970, "avaliacao_2026": 4551},
    "TUO": {"treino": 84744, "validacao": 7837, "teste": 7970, "avaliacao_2026": 4551},
    "TUOV": {"treino": 84720, "validacao": 7781, "teste": 7970, "avaliacao_2026": 4551},
}

CONFIGURACOES_EXECUTAR = ["TU", "TUO", "TUOV"]
DTYPE_NUMERICO = np.float32

DIRETORIO_SAIDA.mkdir(parents=True, exist_ok=True)

print("Arquivo de entrada:", CAMINHO_DADOS.resolve())
print("Diretório de saída:", DIRETORIO_SAIDA.resolve())
print("Configurações:", CONFIGURACOES_EXECUTAR)

Arquivo de entrada: C:\programas\univesp\pic-iv\dados_inmet_consolidados_2026-09-20_16-30.csv
Diretório de saída: C:\programas\univesp\pic-iv\dados_processados_multivariados
Configurações: ['TU', 'TUO', 'TUOV']


## 2. Funções utilitárias

As funções abaixo validam a estrutura, criam atributos temporais, ajustam os normalizadores e constroem as janelas sem cruzar anos ou subconjuntos.

In [4]:
def sha256_arquivo(caminho: Path, tamanho_bloco: int = 1024 * 1024) -> str:
    resumo = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        while bloco := arquivo.read(tamanho_bloco):
            resumo.update(bloco)
    return resumo.hexdigest()


def adicionar_atributos_temporais(dados: pd.DataFrame) -> pd.DataFrame:
    dados = dados.copy()
    hora = dados["datetime"].dt.hour
    dia_ano = dados["datetime"].dt.dayofyear

    dados["hora_sin"] = np.sin(2 * np.pi * hora / 24.0)
    dados["hora_cos"] = np.cos(2 * np.pi * hora / 24.0)
    dados["dia_ano_sin"] = np.sin(2 * np.pi * dia_ano / 365.25)
    dados["dia_ano_cos"] = np.cos(2 * np.pi * dia_ano / 365.25)
    return dados


def validar_grade_temporal(dados: pd.DataFrame) -> None:
    if dados["datetime"].isna().any():
        raise ValueError("Existem timestamps inválidos em 'datetime'.")
    if dados["datetime"].duplicated().any():
        duplicados = dados.loc[dados["datetime"].duplicated(), "datetime"].head().tolist()
        raise ValueError(f"Existem timestamps duplicados. Exemplos: {duplicados}")
    if not dados["datetime"].is_monotonic_increasing:
        raise ValueError("Os timestamps não estão em ordem crescente.")

    diferencas = dados["datetime"].diff().dropna()
    lacunas = diferencas[diferencas != pd.Timedelta(hours=1)]
    if not lacunas.empty:
        raise ValueError(f"A grade não é integralmente horária. Foram encontradas {len(lacunas)} descontinuidades.")


def validar_colunas(dados: pd.DataFrame) -> None:
    obrigatorias = {"datetime", *COLUNAS_ALVO, *sum(CONJUNTOS.values(), [])}
    ausentes = sorted(obrigatorias.difference(dados.columns))
    if ausentes:
        raise ValueError(f"Colunas obrigatórias ausentes: {ausentes}")


def ajustar_normalizadores(dados_treino: pd.DataFrame, colunas_meteorologicas: list[str]):
    # O ajuste usa apenas linhas de treino completas para o respectivo grupo de colunas.
    linhas_x = dados_treino[colunas_meteorologicas].dropna()
    linhas_y = dados_treino[COLUNAS_ALVO].dropna()

    if linhas_x.empty or linhas_y.empty:
        raise ValueError("Não há dados de treino suficientes para ajustar os normalizadores.")

    scaler_x = StandardScaler().fit(linhas_x)
    scaler_y = StandardScaler().fit(linhas_y)
    return scaler_x, scaler_y, len(linhas_x), len(linhas_y)


def maior_sequencia_verdadeira(mascara: np.ndarray) -> int:
    maior = atual = 0
    for valor in mascara:
        atual = atual + 1 if valor else 0
        maior = max(maior, atual)
    return maior


def construir_janelas_ano(
    dados_ano: pd.DataFrame,
    colunas_meteorologicas: list[str],
    scaler_x: StandardScaler,
    scaler_y: StandardScaler,
):
    dados_ano = dados_ano.sort_values("datetime").reset_index(drop=True)

    diferencas = dados_ano["datetime"].diff().dropna()
    if not (diferencas == pd.Timedelta(hours=1)).all():
        raise ValueError(f"O ano {dados_ano['ano'].iloc[0]} não possui grade horária contínua.")

    meteo_bruto = dados_ano[colunas_meteorologicas].to_numpy(dtype=np.float64)
    alvos_brutos = dados_ano[COLUNAS_ALVO].to_numpy(dtype=np.float64)
    temporal = dados_ano[COLUNAS_TEMPORAIS].to_numpy(dtype=DTYPE_NUMERICO)

    # Transformações mantêm NaN, que será rejeitado pela máscara de validade.
    meteo_norm = scaler_x.transform(meteo_bruto).astype(DTYPE_NUMERICO)
    alvos_norm = scaler_y.transform(alvos_brutos).astype(DTYPE_NUMERICO)

    n = len(dados_ano)
    origens = np.arange(TAU, n - HORIZONTE + 1)

    historico_completo = np.isfinite(meteo_norm).all(axis=1)
    futuro_completo = np.isfinite(alvos_norm).all(axis=1)

    # Soma cumulativa para testar blocos de 24 horas sem loops sobre todos os valores.
    soma_hist = np.concatenate(([0], np.cumsum(historico_completo.astype(np.int64))))
    soma_fut = np.concatenate(([0], np.cumsum(futuro_completo.astype(np.int64))))

    validas_hist = (soma_hist[origens] - soma_hist[origens - TAU]) == TAU
    validas_fut = (soma_fut[origens + HORIZONTE] - soma_fut[origens]) == HORIZONTE
    origens_validas = origens[validas_hist & validas_fut]

    n_amostras = len(origens_validas)
    n_atrib_hist = len(colunas_meteorologicas) + len(COLUNAS_TEMPORAIS)

    X_historico = np.empty((n_amostras, TAU, n_atrib_hist), dtype=DTYPE_NUMERICO)
    X_futuro_temporal = np.empty((n_amostras, HORIZONTE, len(COLUNAS_TEMPORAIS)), dtype=DTYPE_NUMERICO)
    y = np.empty((n_amostras, HORIZONTE, len(COLUNAS_ALVO)), dtype=DTYPE_NUMERICO)

    datas = dados_ano["datetime"].to_numpy(dtype="datetime64[ns]")
    inicio_historico = np.empty(n_amostras, dtype="datetime64[ns]")
    fim_historico = np.empty(n_amostras, dtype="datetime64[ns]")
    inicio_previsao = np.empty(n_amostras, dtype="datetime64[ns]")
    fim_previsao = np.empty(n_amostras, dtype="datetime64[ns]")

    for i, origem in enumerate(origens_validas):
        fatia_hist = slice(origem - TAU, origem)
        fatia_fut = slice(origem, origem + HORIZONTE)

        X_historico[i, :, :len(colunas_meteorologicas)] = meteo_norm[fatia_hist]
        X_historico[i, :, len(colunas_meteorologicas):] = temporal[fatia_hist]
        X_futuro_temporal[i] = temporal[fatia_fut]
        y[i] = alvos_norm[fatia_fut]

        inicio_historico[i] = datas[origem - TAU]
        fim_historico[i] = datas[origem - 1]
        inicio_previsao[i] = datas[origem]
        fim_previsao[i] = datas[origem + HORIZONTE - 1]

    return {
        "X_historico": X_historico,
        "X_futuro_temporal": X_futuro_temporal,
        "y": y,
        "inicio_historico": inicio_historico,
        "fim_historico": fim_historico,
        "inicio_previsao": inicio_previsao,
        "fim_previsao": fim_previsao,
    }


def concatenar_partes(partes: list[dict]) -> dict:
    if not partes:
        raise ValueError("Nenhuma parte foi gerada.")
    return {chave: np.concatenate([parte[chave] for parte in partes], axis=0) for chave in partes[0]}


def construir_janelas_split(
    dados_split: pd.DataFrame,
    colunas_meteorologicas: list[str],
    scaler_x: StandardScaler,
    scaler_y: StandardScaler,
) -> dict:
    partes = []
    for ano, dados_ano in dados_split.groupby("ano", sort=True):
        parte = construir_janelas_ano(dados_ano, colunas_meteorologicas, scaler_x, scaler_y)
        partes.append(parte)
    return concatenar_partes(partes)


def validar_janelas(janelas: dict, nome_split: str, anos_permitidos: set[int]) -> None:
    Xh = janelas["X_historico"]
    Xf = janelas["X_futuro_temporal"]
    y = janelas["y"]

    if not np.isfinite(Xh).all() or not np.isfinite(Xf).all() or not np.isfinite(y).all():
        raise AssertionError(f"O split {nome_split} contém NaN ou infinito.")
    if Xh.shape[1] != TAU or Xf.shape[1] != HORIZONTE or y.shape[1:] != (HORIZONTE, len(COLUNAS_ALVO)):
        raise AssertionError(f"Formas incorretas no split {nome_split}: {Xh.shape}, {Xf.shape}, {y.shape}")

    inicio_hist = pd.to_datetime(janelas["inicio_historico"], utc=True)
    fim_hist = pd.to_datetime(janelas["fim_historico"], utc=True)
    inicio_prev = pd.to_datetime(janelas["inicio_previsao"], utc=True)
    fim_prev = pd.to_datetime(janelas["fim_previsao"], utc=True)

    if not set(inicio_prev.year).issubset(anos_permitidos):
        raise AssertionError(f"O split {nome_split} contém anos não permitidos: {sorted(set(inicio_prev.year))}")
    if not ((fim_hist + pd.Timedelta(hours=1)) == inicio_prev).all():
        raise AssertionError(f"Há descontinuidade entre histórico e previsão em {nome_split}.")
    if not ((inicio_prev + pd.Timedelta(hours=HORIZONTE - 1)) == fim_prev).all():
        raise AssertionError(f"Horizonte final incorreto em {nome_split}.")
    if not (inicio_hist.year == fim_prev.year).all():
        raise AssertionError(f"Há janelas cruzando anos em {nome_split}.")


def salvar_split(caminho: Path, janelas: dict) -> None:
    np.savez_compressed(caminho, **janelas)

## 3. Carregamento e validação mínima do consolidado

A auditoria completa já foi realizada anteriormente. Aqui são repetidas somente as verificações necessárias para impedir o processamento silencioso de um arquivo incompatível.

In [5]:
if not CAMINHO_DADOS.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {CAMINHO_DADOS.resolve()}\n"
        "Ajuste CAMINHO_DADOS na seção de configuração."
    )

dados_completos = pd.read_csv(
    CAMINHO_DADOS,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False,
)

validar_colunas(dados_completos)

dados_completos["datetime"] = pd.to_datetime(
    dados_completos["datetime"],
    utc=True,
    errors="coerce",
)
dados_completos = dados_completos.sort_values("datetime").reset_index(drop=True)
validar_grade_temporal(dados_completos)

for coluna in sorted(set(sum(CONJUNTOS.values(), []) + COLUNAS_ALVO)):
    dados_completos[coluna] = pd.to_numeric(dados_completos[coluna], errors="coerce")

dados_completos["ano"] = dados_completos["datetime"].dt.year

diagnostico_inicial = pd.Series({
    "registros": len(dados_completos),
    "inicio": dados_completos["datetime"].min(),
    "fim": dados_completos["datetime"].max(),
    "timestamps_duplicados": int(dados_completos["datetime"].duplicated().sum()),
    "sha256_arquivo": sha256_arquivo(CAMINHO_DADOS),
})

diagnostico_inicial

registros                                                           206640
inicio                                           2003-02-04 00:00:00+00:00
fim                                              2026-08-31 23:00:00+00:00
timestamps_duplicados                                                    0
sha256_arquivo           6cf25143f6ef3ebe8b1e2eff142dc6f200326342332f14...
dtype: object

## 4. Recorte temporal e atributos determinísticos

O ano de 2022 permanece no objeto completo para rastreabilidade, mas não participa do treino, do ajuste dos normalizadores nem da geração das janelas de treinamento.

In [6]:
dados_modelagem = dados_completos.loc[dados_completos["ano"] >= 2011].copy()
dados_modelagem = adicionar_atributos_temporais(dados_modelagem)

splits_df = {
    "treino": dados_modelagem.loc[dados_modelagem["ano"].isin(ANOS_TREINO)].copy(),
    "validacao": dados_modelagem.loc[dados_modelagem["ano"] == ANO_VALIDACAO].copy(),
    "teste": dados_modelagem.loc[dados_modelagem["ano"] == ANO_TESTE].copy(),
    "avaliacao_2026": dados_modelagem.loc[dados_modelagem["ano"] == ANO_AVALIACAO_ADICIONAL].copy(),
}

assert 2022 not in set(splits_df["treino"]["ano"])
assert set(splits_df["validacao"]["ano"]) == {2024}
assert set(splits_df["teste"]["ano"]) == {2025}
assert set(splits_df["avaliacao_2026"]["ano"]) == {2026}

resumo_splits = pd.DataFrame({
    nome: {
        "registros": len(df_split),
        "inicio": df_split["datetime"].min(),
        "fim": df_split["datetime"].max(),
        "anos": ", ".join(map(str, sorted(df_split["ano"].unique()))),
    }
    for nome, df_split in splits_df.items()
}).T

resumo_splits

,registros,inicio,fim,anos
treino,105192,2011-01-01 00:00:00+00:00,2023-12-31 23:00:00+00:00,"2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018..."
validacao,8784,2024-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00,2024
teste,8760,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,2025
avaliacao_2026,5832,2026-01-01 00:00:00+00:00,2026-08-31 23:00:00+00:00,2026


## 5. Geração dos artefatos

Cada configuração recebe normalizadores próprios, arquivos compactados por split e metadados suficientes para reprodução e uso posterior na API.

In [7]:
resultados_contagem = []

for nome_configuracao in CONFIGURACOES_EXECUTAR:
    if nome_configuracao not in CONJUNTOS:
        raise KeyError(f"Configuração desconhecida: {nome_configuracao}")

    colunas_meteorologicas = CONJUNTOS[nome_configuracao]
    diretorio_config = DIRETORIO_SAIDA / nome_configuracao
    diretorio_config.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Preparando {nome_configuracao} ===")
    scaler_x, scaler_y, n_fit_x, n_fit_y = ajustar_normalizadores(
        splits_df["treino"],
        colunas_meteorologicas,
    )

    if scaler_x.n_features_in_ != len(colunas_meteorologicas):
        raise AssertionError("Número de atributos de scaler_x incompatível.")
    if scaler_y.n_features_in_ != len(COLUNAS_ALVO):
        raise AssertionError("Número de atributos de scaler_y incompatível.")

    joblib.dump(scaler_x, diretorio_config / "scaler_x.joblib")
    joblib.dump(scaler_y, diretorio_config / "scaler_y.joblib")

    contagens_config = {}

    for nome_split, df_split in splits_df.items():
        janelas = construir_janelas_split(
            df_split,
            colunas_meteorologicas,
            scaler_x,
            scaler_y,
        )

        anos_permitidos = {
            "treino": set(ANOS_TREINO),
            "validacao": {ANO_VALIDACAO},
            "teste": {ANO_TESTE},
            "avaliacao_2026": {ANO_AVALIACAO_ADICIONAL},
        }[nome_split]
        validar_janelas(janelas, nome_split, anos_permitidos)

        n_amostras = len(janelas["y"])
        esperado = CONTAGENS_ESPERADAS[nome_configuracao][nome_split]
        if n_amostras != esperado:
            raise AssertionError(
                f"Contagem divergente em {nome_configuracao}/{nome_split}: "
                f"obtido={n_amostras}, esperado={esperado}."
            )

        salvar_split(diretorio_config / f"{nome_split}.npz", janelas)
        contagens_config[nome_split] = n_amostras

        resultados_contagem.append({
            "configuracao": nome_configuracao,
            "split": nome_split,
            "amostras": n_amostras,
            "X_historico": str(janelas["X_historico"].shape),
            "X_futuro_temporal": str(janelas["X_futuro_temporal"].shape),
            "y": str(janelas["y"].shape),
        })
        print(nome_split, n_amostras, janelas["X_historico"].shape, janelas["y"].shape)

        del janelas

    metadados = {
        "versao_preparacao": "1.0.0",
        "arquivo_origem": str(CAMINHO_DADOS),
        "sha256_arquivo_origem": diagnostico_inicial["sha256_arquivo"],
        "configuracao": nome_configuracao,
        "tau": TAU,
        "horizonte": HORIZONTE,
        "colunas_meteorologicas_historicas": colunas_meteorologicas,
        "colunas_temporais_historicas": COLUNAS_TEMPORAIS,
        "colunas_futuras_permitidas": COLUNAS_FUTURAS_PERMITIDAS,
        "colunas_alvo": COLUNAS_ALVO,
        "ordem_alvos": {"temperatura": 0, "umidade": 1},
        "anos_treino": ANOS_TREINO,
        "anos_excluidos": ANOS_EXCLUIDOS,
        "ano_validacao": ANO_VALIDACAO,
        "ano_teste": ANO_TESTE,
        "ano_avaliacao_adicional": ANO_AVALIACAO_ADICIONAL,
        "usa_covariaveis_meteorologicas_futuras": False,
        "normalizacao": "StandardScaler ajustado exclusivamente no treino",
        "atributos_temporais_padronizados": False,
        "n_registros_fit_scaler_x": n_fit_x,
        "n_registros_fit_scaler_y": n_fit_y,
        "media_scaler_x": scaler_x.mean_.tolist(),
        "escala_scaler_x": scaler_x.scale_.tolist(),
        "media_scaler_y": scaler_y.mean_.tolist(),
        "escala_scaler_y": scaler_y.scale_.tolist(),
        "contagens": contagens_config,
        "dtype_tensores": str(np.dtype(DTYPE_NUMERICO)),
        "versoes": {
            "python": sys.version.split()[0],
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "plataforma": platform.platform(),
        },
    }

    with (diretorio_config / "metadados.json").open("w", encoding="utf-8") as arquivo:
        json.dump(metadados, arquivo, ensure_ascii=False, indent=2)

resumo_contagens = pd.DataFrame(resultados_contagem)
resumo_contagens


=== Preparando TU ===


c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-pack

treino 84744 (84744, 24, 6) (84744, 24, 2)
validacao 7837 (7837, 24, 6) (7837, 24, 2)


c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


teste 7970 (7970, 24, 6) (7970, 24, 2)
avaliacao_2026 4551 (4551, 24, 6) (4551, 24, 2)

=== Preparando TUO ===


...

validacao 7781 (7781, 24, 8) (7781, 24, 2)
teste 7970 (7970, 24, 8) (7970, 24, 2)


c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


avaliacao_2026 4551 (4551, 24, 8) (4551, 24, 2)


c:\Users\adilson.pas\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,configuracao,split,amostras,X_historico,X_futuro_temporal,y
0,TU,treino,84744,"(84744, 24, 6)","(84744, 24, 4)","(84744, 24, 2)"
1,TU,validacao,7837,"(7837, 24, 6)","(7837, 24, 4)","(7837, 24, 2)"
2,TU,teste,7970,"(7970, 24, 6)","(7970, 24, 4)","(7970, 24, 2)"
3,TU,avaliacao_2026,4551,"(4551, 24, 6)","(4551, 24, 4)","(4551, 24, 2)"
4,TUO,treino,84744,"(84744, 24, 7)","(84744, 24, 4)","(84744, 24, 2)"
5,TUO,validacao,7837,"(7837, 24, 7)","(7837, 24, 4)","(7837, 24, 2)"
6,TUO,teste,7970,"(7970, 24, 7)","(7970, 24, 4)","(7970, 24, 2)"
7,TUO,avaliacao_2026,4551,"(4551, 24, 7)","(4551, 24, 4)","(4551, 24, 2)"
8,TUOV,treino,84720,"(84720, 24, 8)","(84720, 24, 4)","(84720, 24, 2)"
9,TUOV,validacao,7781,"(7781, 24, 8)","(7781, 24, 4)","(7781, 24, 2)"


## 6. Verificação independente dos arquivos salvos

Esta etapa reabre os artefatos e confirma as formas, a ausência de valores inválidos e a impossibilidade de covariáveis meteorológicas futuras.

In [8]:
verificacao_final = []

for nome_configuracao in CONFIGURACOES_EXECUTAR:
    diretorio_config = DIRETORIO_SAIDA / nome_configuracao

    with (diretorio_config / "metadados.json").open("r", encoding="utf-8") as arquivo:
        metadados = json.load(arquivo)

    if metadados["usa_covariaveis_meteorologicas_futuras"] is not False:
        raise AssertionError("Metadados indicam uso indevido de covariáveis meteorológicas futuras.")
    if metadados["colunas_futuras_permitidas"] != COLUNAS_FUTURAS_PERMITIDAS:
        raise AssertionError("Lista de covariáveis futuras diverge do protocolo aprovado.")

    for nome_split in splits_df:
        caminho = diretorio_config / f"{nome_split}.npz"
        with np.load(caminho) as artefato:
            chaves_esperadas = {
                "X_historico", "X_futuro_temporal", "y",
                "inicio_historico", "fim_historico", "inicio_previsao", "fim_previsao",
            }
            if set(artefato.files) != chaves_esperadas:
                raise AssertionError(f"Chaves inesperadas em {caminho}: {artefato.files}")

            n = artefato["y"].shape[0]
            esperado = CONTAGENS_ESPERADAS[nome_configuracao][nome_split]
            valido = (
                n == esperado
                and np.isfinite(artefato["X_historico"]).all()
                and np.isfinite(artefato["X_futuro_temporal"]).all()
                and np.isfinite(artefato["y"]).all()
            )
            verificacao_final.append({
                "configuracao": nome_configuracao,
                "split": nome_split,
                "amostras": n,
                "arquivo_mb": round(caminho.stat().st_size / 1024**2, 2),
                "valido": valido,
            })

verificacao_final = pd.DataFrame(verificacao_final)
if not verificacao_final["valido"].all():
    raise AssertionError("Uma ou mais verificações finais falharam.")

verificacao_final

,configuracao,split,amostras,arquivo_mb,valido
0,TU,treino,84744,3.95,True
1,TU,validacao,7837,0.37,True
2,TU,teste,7970,0.37,True
3,TU,avaliacao_2026,4551,0.21,True
4,TUO,treino,84744,4.21,True
5,TUO,validacao,7837,0.39,True
6,TUO,teste,7970,0.40,True
7,TUO,avaliacao_2026,4551,0.22,True
8,TUOV,treino,84720,4.41,True
9,TUOV,validacao,7781,0.41,True


## 7. Manifesto geral

O manifesto facilita a localização dos artefatos pelo notebook de treinamento e pela futura API.

In [9]:
manifesto = {
    "versao": "1.0.0",
    "diretorio_raiz": str(DIRETORIO_SAIDA),
    "configuracoes": CONFIGURACOES_EXECUTAR,
    "splits": list(splits_df),
    "tau": TAU,
    "horizonte": HORIZONTE,
    "alvos": COLUNAS_ALVO,
    "contagens_esperadas": CONTAGENS_ESPERADAS,
    "arquivo_origem": str(CAMINHO_DADOS),
    "sha256_arquivo_origem": diagnostico_inicial["sha256_arquivo"],
}

with (DIRETORIO_SAIDA / "manifesto.json").open("w", encoding="utf-8") as arquivo:
    json.dump(manifesto, arquivo, ensure_ascii=False, indent=2)

print("Preparação concluída.")
print("Manifesto:", (DIRETORIO_SAIDA / "manifesto.json").resolve())

Preparação concluída.
Manifesto: C:\programas\univesp\pic-iv\dados_processados_multivariados\manifesto.json


## Critério de encerramento

A preparação está concluída quando todas as células terminarem sem exceção e a tabela de verificação final apresentar `valido = True` para os quatro splits das três configurações.

O próximo notebook poderá carregar diretamente os arquivos `.npz`, sem reler ou reprocessar todo o CSV consolidado.